In [7]:
!pip install -q -U google-generativeai

In [8]:
import google.generativeai as genai

API_KEY = "AQ.Ab8RN6LTCnUoK-SPoe6O7jdcQFrVH9cVAPrHJ9ZPM_qV_LiBRQ"


genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

In [9]:
schema = """
Table: Customers

customer_id
customer_name
city

Table: Orders

order_id
customer_id
product
amount
order_date
"""

print(schema)


Table: Customers

customer_id
customer_name
city

Table: Orders

order_id
customer_id
product
amount
order_date



In [10]:

user_question = """
Show all customers who spent more than 5000.
"""

In [11]:
prompt = f"""
You are an expert SQL Developer.

Database Schema:

{schema}

User Request:

{user_question}

Generate only the SQL query.
"""

In [12]:
response = model.generate_content(prompt)

generated_sql = response.text

print(generated_sql)

```sql
SELECT
  c.customer_id,
  c.customer_name,
  c.city
FROM Customers AS c
JOIN Orders AS o
  ON c.customer_id = o.customer_id
GROUP BY
  c.customer_id,
  c.customer_name,
  c.city
HAVING
  SUM(o.amount) > 5000;
```


In [13]:
with open(
    "generated_sql.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(generated_sql)

print("SQL Saved")

SQL Saved


In [14]:
explanation_prompt = f"""
You are an SQL instructor.

Explain the following SQL query in simple language.

SQL Query:

{generated_sql}

Explain:
1. What the query does
2. Tables used
3. Conditions used
4. Expected output
"""

In [15]:
explanation_response = model.generate_content(
    explanation_prompt
)

sql_explanation = explanation_response.text

print(sql_explanation)

Alright class, let's break down this SQL query step by step! It's a great example of how we combine data, group it, and then filter those groups based on an aggregate calculation.

---

### SQL Query:

```sql
SELECT
  c.customer_id,
  c.customer_name,
  c.city
FROM Customers AS c
JOIN Orders AS o
  ON c.customer_id = o.customer_id
GROUP BY
  c.customer_id,
  c.customer_name,
  c.city
HAVING
  SUM(o.amount) > 5000;
```

---

### 1. What the query does

In simple terms, this query is designed to find our **"high-spending" customers**.

It identifies all customers who, when you add up the total amount of *all* their orders, have spent more than 5000 (whatever currency unit that amount represents). For these high-spending customers, it then shows us their customer ID, name, and the city they live in.

Think of it as: "Show me the ID, name, and city of every customer whose *total lifetime order value* is over 5000."

### 2. Tables used

This query uses two tables:

*   **`Customers` (aliase

In [16]:
with open(
    "sql_explanation.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(sql_explanation)

print("SQL Explanation Saved")

SQL Explanation Saved


In [17]:
optimization_prompt = f"""
You are a Senior SQL Performance Engineer.

Analyze the following SQL query.

SQL Query:

{generated_sql}

Provide:

1. Performance Issues
2. Optimization Suggestions
3. Index Recommendations
4. Better Version of Query
"""

In [18]:
optimization_response = model.generate_content(
    optimization_prompt
)

optimization_report = optimization_response.text

print(optimization_report)

As a Senior SQL Performance Engineer, I've analyzed the provided query to identify potential bottlenecks and propose improvements.

---

### SQL Query Under Analysis:

```sql
SELECT
  c.customer_id,
  c.customer_name,
  c.city
FROM Customers AS c
JOIN Orders AS o
  ON c.customer_id = o.customer_id
GROUP BY
  c.customer_id,
  c.customer_name,
  c.city
HAVING
  SUM(o.amount) > 5000;
```

---

### 1. Performance Issues

1.  **Large Intermediate Result Set (Pre-Aggregation Join)**: The query first performs a `JOIN` between `Customers` and `Orders`. If a customer has many orders, this join will produce many rows for that customer (one row for each order). The `GROUP BY` then operates on this potentially very large intermediate result set, which can be memory-intensive and lead to costly disk spills if the data doesn't fit in memory.
2.  **Resource-Intensive `GROUP BY` Operation**: The `GROUP BY` is performed on three columns (`c.customer_id`, `c.customer_name`, `c.city`). Although `c.custom

In [19]:
with open(
    "sql_optimization_report.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(optimization_report)

print("Optimization Report Saved")

Optimization Report Saved


In [20]:
review_prompt = f"""
You are an SQL Code Reviewer.

Review the following SQL query.

SQL Query:

{generated_sql}

Provide:

1. Code Quality Score (out of 10)
2. Readability Review
3. Maintainability Review
4. Best Practice Suggestions
"""

In [21]:
review_response = model.generate_content(
    review_prompt
)

sql_review = review_response.text

print(sql_review)

This is a well-structured and clear SQL query. Let's break down the review.

---

### 1. Code Quality Score (out of 10)

**Score: 9/10**

This query is highly effective and demonstrates good practices. It achieves its goal clearly and efficiently for the given task. The only minor deductions would be for potential performance considerations on very large datasets or a slight ambiguity in intent regarding customers without orders (though the current implementation is correct for its `HAVING` clause).

---

### 2. Readability Review

**Excellent.**

*   **Formatting:** Consistent capitalization for SQL keywords (`SELECT`, `FROM`, `JOIN`, `ON`, `GROUP BY`, `HAVING`). Clear indentation for columns and clauses, making the query flow easy to follow.
*   **Aliases:** Tables are aliased (`Customers AS c`, `Orders AS o`), and these aliases are consistently used throughout the query (`c.customer_id`, `o.amount`), which improves brevity and clarity, especially when dealing with potential column n

In [22]:
with open(
    "sql_review.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(sql_review)

print("SQL Review Saved")

SQL Review Saved


In [23]:
sql_query = """
SELECT customer_name
FROM Customers
WHERE amount > 5000
"""

In [24]:
error_prompt = f"""
You are an SQL expert.

Analyze the following SQL query.

Query:

{sql_query}

Provide:

1. Errors in the query
2. Why they occur
3. Correct version of query
"""

In [25]:
error_response = model.generate_content(error_prompt)

error_report = error_response.text

print(error_report)

As an SQL expert, I've analyzed your query.

The query you provided is **syntactically correct** according to standard SQL. This means its structure follows the rules of the SQL language.

However, a query's "correctness" in practice also depends heavily on the **database schema** it's executed against. Without knowing the actual tables and columns in your database, we can only identify potential runtime errors.

---

### 1. Errors in the query

There are **no direct syntax errors** in the provided SQL query.

However, there are potential **runtime errors** that would occur if the database schema does not match the query's expectations:

1.  **Table Not Found Error:** If a table named `Customers` does not exist in the database or the currently selected schema.
2.  **Column Not Found Error (customer_name):** If the `Customers` table exists, but it does not have a column named `customer_name`.
3.  **Column Not Found Error (amount):** If the `Customers` table exists, but it does not have 

In [26]:
with open(
    "sql_error_report.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(error_report)

print("Error Report Saved")

Error Report Saved


In [27]:
practice_prompt = """
Generate:

1. 10 Beginner SQL Questions
2. 10 Intermediate SQL Questions
3. 5 Advanced SQL Questions

Include answers.
"""

In [28]:
practice_response = model.generate_content(
    practice_prompt
)

practice_questions = practice_response.text

print(practice_questions)

Okay, here are SQL questions categorized by difficulty, along with their answers.

First, let's establish a sample database schema we'll use for these questions:

---

## Sample Database Schema

We'll use the following tables:

1.  **`Customers`**
    *   `customer_id` (INT, Primary Key)
    *   `first_name` (VARCHAR)
    *   `last_name` (VARCHAR)
    *   `email` (VARCHAR, UNIQUE)
    *   `registration_date` (DATE)
    *   `city` (VARCHAR)
    *   `country` (VARCHAR)

2.  **`Products`**
    *   `product_id` (INT, Primary Key)
    *   `product_name` (VARCHAR)
    *   `category` (VARCHAR)
    *   `price` (DECIMAL)
    *   `stock_quantity` (INT)

3.  **`Orders`**
    *   `order_id` (INT, Primary Key)
    *   `customer_id` (INT, Foreign Key referencing `Customers`)
    *   `order_date` (DATE)
    *   `total_amount` (DECIMAL)
    *   `status` (VARCHAR - e.g., 'Pending', 'Completed', 'Cancelled')

4.  **`Order_Items`**
    *   `item_id` (INT, Primary Key)
    *   `order_id` (INT, Foreign Key

In [29]:
with open(
    "sql_practice_questions.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(practice_questions)

print("Questions Saved")

Questions Saved


In [30]:
interview_prompt = """
You are a Senior Data Analyst interviewer.

Generate:

1. Top 20 SQL Interview Questions
2. Detailed Answers
3. Difficulty Level
"""

In [31]:
interview_response = model.generate_content(
    interview_prompt
)

interview_questions = interview_response.text

print(interview_questions)

Alright, let's get you prepared for that Senior Data Analyst interview. My goal here is not just to test SQL syntax, but your understanding of data, performance, and problem-solving using SQL.

Here's a set of 20 questions, covering a range of topics from fundamental to advanced, along with detailed answers, key concepts, and common pitfalls.

---

### **Interviewer's Note:**

"Welcome! As a Senior Data Analyst, you'll be spending a significant amount of time working with data, often through SQL. These questions are designed to assess your technical proficiency, your ability to think critically about data problems, and your understanding of best practices. Don't just provide a query; explain your thought process, potential alternatives, and any performance considerations."

---

### **Schema for Questions:**

For consistency, let's assume the following simplified schema:

*   **`Employees`**
    *   `employee_id` (PK, INT)
    *   `first_name` (VARCHAR)
    *   `last_name` (VARCHAR)
  

In [32]:
with open(
    "sql_interview_guide.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(interview_questions)

print("Interview Guide Saved")

Interview Guide Saved


In [32]:
schema_prompt = """
Generate a database schema for an E-Commerce system.

Include:

1. Table names
2. Columns
3. Primary Keys
4. Foreign Keys

Return in a clean format.
"""

In [34]:
schema_response = model.generate_content(
    "schema_prompt"
)

generated_schema = schema_response.text

print(generated_schema)

A "schema prompt" can refer to a few things, but most commonly it's a prompt designed to either:

1.  **Generate a schema:** Asking an AI to create a data schema based on your requirements.
2.  **Utilize a schema:** Providing a schema to an AI and asking it to perform a task (e.g., validate data, generate data, write code, explain it).

Here are templates for both scenarios:

---

## 1. Prompt to **GENERATE** a Schema

Use this when you want the AI to create a schema (e.g., JSON Schema, OpenAPI, SQL DDL, GraphQL, Pydantic, TypeScript) based on your description.

### Template Structure:

```
**Goal:** Generate a [Target Schema Format, e.g., JSON Schema, OpenAPI YAML, PostgreSQL DDL, TypeScript Interface, Python Pydantic Model] for a [Purpose/Context, e.g., REST API resource, database table, configuration file, UI form validation].

**Entity/Object Name:** [The primary entity the schema describes, e.g., "Product", "User", "OrderRequest"]

**Fields/Properties:**
List each field/property w

In [35]:
with open(
    "database_schema.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(generated_schema)

print("Schema Saved")

Schema Saved


In [36]:
cheat_sheet_prompt = """
Create a SQL Cheat Sheet.

Include:

1. SELECT
2. WHERE
3. ORDER BY
4. GROUP BY
5. HAVING
6. JOIN
7. LEFT JOIN
8. RIGHT JOIN
9. Subqueries
10. Window Functions

Include syntax and examples.
"""

In [37]:
cheat_response = model.generate_content(
    cheat_sheet_prompt
)

sql_cheat_sheet = cheat_response.text

print(sql_cheat_sheet)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5233.42ms


This SQL Cheat Sheet covers essential SQL concepts, providing syntax and examples for common database operations.

---

## SQL Cheat Sheet

### Sample Schema for Examples

To make the examples consistent, let's assume we have two tables:

**1. `Employees` Table**
| EmployeeID (PK) | FirstName | LastName | DepartmentID (FK) | Salary | HireDate |
|-----------------|-----------|----------|-------------------|--------|----------|
| 101             | Alice     | Smith    | 1                 | 75000  | 2020-01-15 |
| 102             | Bob       | Johnson  | 2                 | 82000  | 2019-03-22 |
| 103             | Charlie   | Brown    | 1                 | 60000  | 2021-06-01 |
| 104             | Diana     | Prince   | 3                 | 95000  | 2018-11-10 |
| 105             | Eve       | Davis    | 2                 | 78000  | 2020-09-01 |
| 106             | Frank     | Miller   | 1                 | 70000  | 2022-02-28 |
| 107             | Grace     | Taylor   | NULL             

In [38]:
with open(
    "sql_cheat_sheet.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(sql_cheat_sheet)

print("Cheat Sheet Saved")

Cheat Sheet Saved


In [39]:
summary_prompt = """
Summarize the capabilities of an AI SQL Assistant Agent.

Mention:

- SQL Generation
- SQL Explanation
- Optimization
- Error Detection
- Interview Preparation

Keep it professional.
"""

In [40]:
summary_response = model.generate_content(
    summary_prompt
)

project_summary = summary_response.text

print(project_summary)

An AI SQL Assistant Agent is a sophisticated tool engineered to streamline and enhance various aspects of SQL development and database interaction. Its core capabilities include:

*   **SQL Generation:** Translates natural language prompts and user intent into accurate, executable SQL queries across different database systems. This includes generating complex queries for data retrieval, manipulation (INSERT, UPDATE, DELETE), and definition (CREATE, ALTER, DROP), significantly speeding up development and enabling non-experts to interact with databases effectively.
*   **SQL Explanation:** Deconstructs existing SQL queries, providing clear, human-readable explanations of their logic, purpose, and how different clauses, functions, and joins interact. This capability demystifies complex code, aids in learning, facilitates onboarding, and simplifies debugging.
*   **Optimization:** Analyzes SQL queries for potential performance bottlenecks and suggests improvements to enhance execution spee